In [8]:
import os
import pandas as pd
import numpy as np

DATA_RAW = "../data_pipeline/data/raw"
DATA_PROC = "../data_pipeline/data/processed"

In [9]:
# read the returns and features into the current notebook.

returns = pd.read_csv(
    os.path.join(DATA_PROC, "daily_returns.csv"),
    index_col="date",
    parse_dates=True
)

features = pd.read_csv(
    os.path.join(DATA_PROC, "features_basic.csv"),
    index_col="date",
    parse_dates=True
)

#show shape and features
print("Returns:", returns.shape)
print("Features:", features.shape)
features.head()

Returns: (1983, 60)
Features: (1954, 180)


,AAPL_mom30,ABBV_mom30,ACN_mom30,ADBE_mom30,AMAT_mom30,AMD_mom30,AMT_mom30,AMZN_mom30,AVGO_mom30,BA_mom30,...,QCOM_vol30,SBUX_vol30,SPGI_vol30,TMO_vol30,TXN_vol30,UNH_vol30,UPS_vol30,V_vol30,WMT_vol30,XOM_vol30
date,,,,,,,,,,,,,,,,,,,,,
2018-02-14,-0.024426,0.157312,0.038676,0.109342,-0.020362,0.111111,-0.040034,0.220385,-0.072469,0.167472,...,0.021554,0.012988,0.024121,0.018103,0.026590,0.019153,0.017180,0.017794,0.013917,0.016891
2018-02-15,0.008508,0.157820,0.042122,0.121133,0.000185,0.055411,-0.038538,0.213885,-0.067054,0.202887,...,0.021780,0.012646,0.024112,0.017943,0.026327,0.019223,0.016883,0.017867,0.014083,0.016338
2018-02-16,0.000595,0.201960,0.040159,0.100098,0.013630,-0.024752,0.002937,0.197670,-0.078254,0.202658,...,0.021810,0.012615,0.024018,0.017849,0.026326,0.019389,0.016758,0.017893,0.014301,0.016379
2018-02-20,-0.013996,0.175218,0.023911,0.095176,0.039618,0.011785,-0.009039,0.194616,-0.080995,0.149768,...,0.021891,0.012460,0.024012,0.017823,0.026329,0.019210,0.016781,0.017399,0.023686,0.016413
2018-02-21,-0.014812,0.193646,0.009626,0.092142,0.005583,-0.045603,-0.032181,0.189314,-0.086862,0.141736,...,0.021940,0.012467,0.023998,0.017888,0.026444,0.018978,0.016466,0.017574,0.023924,0.016384


In [10]:

#on a given day we want to calculate what our total returns will be over 21 days from that day. 

horizon = 21

# Compute cumulative forward returns for each stock
fwd_returns = (1 + returns).rolling(window=horizon).apply(lambda x: np.prod(x) - 1, raw=True).shift(-horizon+1)
fwd_returns.head()

,AAPL,ABBV,ACN,ADBE,AMAT,AMD,AMT,AMZN,AVGO,BA,...,QCOM,SBUX,SPGI,TMO,TXN,UNH,UPS,V,WMT,XOM
date,,,,,,,,,,,,,,,,,,,,,
2018-01-03,-0.026007,0.190677,0.043031,0.122003,0.004525,0.206740,0.042018,0.169040,-0.105689,0.202466,...,0.024540,-0.028284,0.082560,0.152244,0.048491,0.063574,-0.033638,0.097895,0.070291,0.047513
2018-01-04,-0.068106,0.160541,0.015205,0.080645,-0.061134,0.077922,0.024986,0.187469,-0.127626,0.171625,...,0.001971,-0.050077,0.038111,0.107724,-0.001606,0.037587,-0.078633,0.045482,0.050578,-0.025029
2018-01-05,-0.095590,0.109836,-0.029096,0.038478,-0.110149,-0.045380,0.017406,0.149150,-0.155248,0.108572,...,-0.065122,-0.071950,-0.040489,0.046889,-0.032351,-0.019737,-0.109487,0.001637,0.005525,-0.081778
2018-01-08,-0.068400,0.107681,-0.018900,0.049261,-0.079670,-0.019360,-0.002420,0.173861,-0.115014,0.103840,...,-0.031142,-0.067102,0.024246,0.035031,-0.024414,-0.015520,-0.116831,0.009339,0.007690,-0.096830
2018-01-09,-0.084944,0.150216,-0.023784,0.039451,-0.129292,-0.055375,-0.014154,0.136269,-0.128145,0.122425,...,-0.018560,-0.076793,0.031098,0.023826,-0.064717,0.004716,-0.133622,0.002598,0.012203,-0.117053


In [11]:
# 1) Find dates where we have *all* three: returns, fwd_returns, features
common_index = (
    returns.index
    .intersection(fwd_returns.index)
    .intersection(features.index)
)

len(common_index), len(returns), len(features)


(1954, 1983, 1954)

In [12]:
# align the indexes so that each data set is using the middle date ranges we care about. 
# we do this because we can't compute momentum on 90 days if we have chunks looking at 1-89 days, etc.
# same for the end data since we don't have info as it ends out. 
# basically only keep the dates where I know what the world looked like (features) and what happened over the next month (forward return).

returns_aligned     = returns.loc[common_index].copy()
fwd_returns_aligned = fwd_returns.loc[common_index].copy()
features_aligned    = features.loc[common_index].copy()

print("Returns aligned:", returns_aligned.shape)
print("Fwd returns aligned:", fwd_returns_aligned.shape)
print("Features aligned:", features_aligned.shape)

Returns aligned: (1954, 60)
Fwd returns aligned: (1954, 60)
Features aligned: (1954, 180)


In [13]:
# build dataset panel to show date, ticker, features, fwd_return

def build_panel(returns_aligned, features_aligned, fwd_returns_aligned):
    rows = []
    tickers = returns_aligned.columns

    for t in tickers:
        prefix = f"{t}_"
        # Only this ticker's factor columns
        t_cols = [c for c in features_aligned.columns if c.startswith(prefix)]
        if not t_cols:
            continue  # skip if no features for this ticker

        # Subset features to this ticker's columns
        feat_t = features_aligned[t_cols].copy()

        # Rename columns: AAPL_mom30 -> mom30, etc.
        feat_t.columns = [c.replace(prefix, "") for c in t_cols]

        # Add ticker & date
        feat_t["ticker"] = t
        feat_t["date"] = feat_t.index

        # Add this ticker's forward return
        feat_t["fwd_ret"] = fwd_returns_aligned[t]

        # Drop rows where we don't know the future return or have missing features
        feat_t = feat_t.dropna(subset=["fwd_ret"])

        rows.append(feat_t)

    panel = pd.concat(rows, ignore_index=True)
    return panel

panel = build_panel(
    returns_aligned,
    features_aligned,
    fwd_returns_aligned
)
panel.shape
panel.sample(20)



,mom30,mom90,vol30,ticker,date,fwd_ret
68327,0.148175,0.396293,0.013603,MA,2020-08-26,-0.055946
83797,0.115037,0.534904,0.022639,NOW,2020-08-24,0.047061
100044,-0.078497,-0.116846,0.010635,SBUX,2023-09-22,0.011708
20577,-0.063779,0.071868,0.016046,BAC,2023-01-13,0.034233
82515,-0.055485,0.111132,0.016523,NKE,2023-03-28,0.051184
55262,-0.102486,-0.324103,0.025334,ISRG,2022-07-14,0.168069
31535,0.089539,0.011559,0.023789,CRM,2020-06-22,0.020594
111061,0.036949,0.098685,0.011315,V,2021-05-24,0.040394
57968,0.131129,0.121661,0.014630,JNJ,2025-08-12,0.018757
28947,0.203806,0.296719,0.011814,CAT,2025-07-28,-0.003343


In [14]:
panel.to_csv(f"{DATA_PROC}/model_panel.csv", index=False)